In [ ]:
import numpy as np
import pandas as pd
import os

In [ ]:
!git clone https://github.com/jenilrupareliya5150-bit/FlyRankAi-ml-Track.git

Cloning into 'FlyRankAi-ml-Track'...
remote: Enumerating objects: 193, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 193 (delta 88), reused 89 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (193/193), 2.67 MiB | 4.03 MiB/s, done.
Resolving deltas: 100% (88/88), done.


In [ ]:
%cd FlyRankAi-ml-Track

/content/FlyRankAi-ml-Track/FlyRankAi-ml-Track


In [ ]:
df=pd.read_csv("data/raw/content_refresh_anonymized.csv")

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jenilrupareliya5150-bit/FlyRankAi-ml-Track/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    precision_recall_curve
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# Use the real warehouse dataset
# Requires your HF_TOKEN in Colab Secrets

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")``

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

print("HF token loaded successfully.")

HF token loaded successfully.


In [3]:
# Load the real FlyRank warehouse daily-performance table
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

print("Real warehouse dataset connected successfully.")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Real warehouse dataset connected successfully.


In [4]:
# Work with 300,000 real rows for this notebook
# The rows are NOT synthetic.

MAX_ROWS = 300_000

rows = []

for i, row in enumerate(ds):
    rows.append(row)

    if i + 1 >= MAX_ROWS:
        break

real_df = pd.DataFrame(rows)

print("Real-data shape:", real_df.shape)
display(real_df.head())

Real-data shape: (300000, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115.0,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358.0,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34.0,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140.0,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89.0,...,0,0,0,0,0,0,0,0,0,0


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

The FlyRank research paper reports findings about search/content performance.

For each selected finding, I ask two questions:

- Where does the label or outcome come from?
- Does the validation design support the claim?

These questions are not meant to reject the findings. They are meant to understand how strongly the evidence supports each conclusion.

In [5]:
# Basic data checks that support the methodology discussion

print("Real warehouse rows:", len(real_df))
print("Number of clients:", real_df["client_hash_id"].nunique())
print("Number of content items:", real_df["content_hash_id"].nunique())

real_df["report_date"] = pd.to_datetime(real_df["report_date"])

print("Minimum date:", real_df["report_date"].min())
print("Maximum date:", real_df["report_date"].max())

Real warehouse rows: 300000
Number of clients: 4
Number of content items: 13928
Minimum date: 2025-01-27 00:00:00
Maximum date: 2025-04-27 00:00:00


### Finding 1: Freshness signals

The paper reports that freshness is associated with a better Health Score, especially for longer content, and suggests that freshness can amplify editing quality rather than replace it.

**Methodology question:** How exactly was freshness measured, and was the Health Score measured using an independent outcome that was not derived from the same freshness-related features?

### Finding 2: Feature importance

The paper reports that some features appear more important than others for explaining or predicting the outcome.

**Methodology question:** Was feature importance evaluated on held-out data, and were correlated or potentially leaky features checked before interpreting the importance results?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


The prediction task is:

Can today's Search Console performance identify content that will receive high Google Search impressions on the following day?

The target is created from the next day's impressions for the same client and content item.

The "before" result uses a random split.

The "after" result uses a time-aware split, where earlier dates are used for training and later dates are used for testing.

The time-aware result is the more realistic estimate for future prediction.

1.   List item

1.   List item
2.   List item


2.   List item



In [6]:
# Sort chronologically before creating the next-day target

df = real_df.copy()

df["report_date"] = pd.to_datetime(df["report_date"])

df = df.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
).reset_index(drop=True)

print("Rows:", len(df))
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

Rows: 300000
Date range: 2025-01-27 00:00:00 to 2025-04-27 00:00:00


In [7]:
# Create next-day observation

group_cols = ["client_hash_id", "content_hash_id"]

df["next_date"] = (
    df.groupby(group_cols)["report_date"]
      .shift(-1)
)

df["next_gsc_impressions"] = (
    df.groupby(group_cols)["gsc_impressions"]
      .shift(-1)
)

# Only accept a true next-day observation
df["next_day_observed"] = (
    df["next_date"] ==
    df["report_date"] + pd.Timedelta(days=1)
)

model_df = df[df["next_day_observed"]].copy()

print("Rows with valid next-day observation:", len(model_df))

display(
    model_df[
        [
            "report_date",
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions",
            "next_date",
            "next_gsc_impressions"
        ]
    ].head(10)
)

Rows with valid next-day observation: 230613


,report_date,client_hash_id,content_hash_id,gsc_impressions,next_date,next_gsc_impressions
0,2025-02-12,client_73cda7b4e4f265ea,content_00033c286cc93446,3,2025-02-13,5.0
1,2025-02-13,client_73cda7b4e4f265ea,content_00033c286cc93446,5,2025-02-14,6.0
2,2025-02-14,client_73cda7b4e4f265ea,content_00033c286cc93446,6,2025-02-15,2.0
3,2025-02-15,client_73cda7b4e4f265ea,content_00033c286cc93446,2,2025-02-16,3.0
4,2025-02-16,client_73cda7b4e4f265ea,content_00033c286cc93446,3,2025-02-17,3.0
5,2025-02-17,client_73cda7b4e4f265ea,content_00033c286cc93446,3,2025-02-18,6.0
6,2025-02-18,client_73cda7b4e4f265ea,content_00033c286cc93446,6,2025-02-19,4.0
7,2025-02-19,client_73cda7b4e4f265ea,content_00033c286cc93446,4,2025-02-20,4.0
8,2025-02-20,client_73cda7b4e4f265ea,content_00033c286cc93446,4,2025-02-21,5.0
9,2025-02-21,client_73cda7b4e4f265ea,content_00033c286cc93446,5,2025-02-22,5.0


In [8]:
# Fixed threshold for the binary prediction target

TARGET_THRESHOLD = 24.0

model_df["target"] = (
    model_df["next_gsc_impressions"] >= TARGET_THRESHOLD
).astype(int)

print("Target threshold:", TARGET_THRESHOLD)

print("\nTarget distribution:")
print(model_df["target"].value_counts())

print("\nTarget proportion:")
print(model_df["target"].value_counts(normalize=True))

Target threshold: 24.0

Target distribution:
target
0    170710
1     59903
Name: count, dtype: int64

Target proportion:
target
0    0.740244
1    0.259756
Name: proportion, dtype: float64


In [9]:
# Features available at today's prediction point.
# We intentionally do NOT use next-day columns.

candidate_features = [
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other",
    "scroll_events"
]

feature_cols = [
    col for col in candidate_features
    if col in model_df.columns
]

print("Number of candidate features:", len(feature_cols))
print(feature_cols)

Number of candidate features: 27
['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [10]:
# Prepare numeric features

X = model_df[feature_cols].copy()
y = model_df["target"].copy()

# Convert boolean columns to integers
for col in X.columns:
    if X[col].dtype == "bool":
        X[col] = X[col].astype(int)

# Convert everything to numeric
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Handle missing/infinite values
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print("Final feature matrix:", X.shape)

Final feature matrix: (230613, 27)


In [11]:
from sklearn.model_selection import train_test_split

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Random training shape:", X_train_random.shape)
print("Random test shape:", X_test_random.shape)

Random training shape: (184490, 27)
Random test shape: (46123, 27)


In [12]:
random_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

random_model.fit(X_train_random, y_train_random)

random_proba = random_model.predict_proba(X_test_random)[:, 1]
random_pred = (random_proba >= 0.5).astype(int)

random_auc = roc_auc_score(
    y_test_random,
    random_proba
)

random_precision = precision_score(
    y_test_random,
    random_pred,
    zero_division=0
)

print("BEFORE — Random split")
print("----------------------")
print("ROC-AUC:", round(random_auc, 4))
print("Precision:", round(random_precision, 4))

BEFORE — Random split
----------------------
ROC-AUC: 0.9498
Precision: 0.741


In [13]:
# Sort by date

model_df = model_df.sort_values("report_date").reset_index(drop=True)

split_date = model_df["report_date"].quantile(0.80)

train_mask = model_df["report_date"] < split_date
test_mask = model_df["report_date"] >= split_date

train_time = model_df.loc[train_mask]
test_time = model_df.loc[test_mask]

X_train_time = train_time[feature_cols].copy()
X_test_time = test_time[feature_cols].copy()

y_train_time = train_time["target"].copy()
y_test_time = test_time["target"].copy()

for col in X_train_time.columns:
    if X_train_time[col].dtype == "bool":
        X_train_time[col] = X_train_time[col].astype(int)
        X_test_time[col] = X_test_time[col].astype(int)

X_train_time = (
    X_train_time
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

X_test_time = (
    X_test_time
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("Time split date:", split_date)
print("Training:", X_train_time.shape)
print("Testing:", X_test_time.shape)

Time split date: 2025-03-25 00:00:00
Training: (181183, 27)
Testing: (49430, 27)


In [14]:
time_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

time_model.fit(
    X_train_time,
    y_train_time
)

time_proba = time_model.predict_proba(X_test_time)[:, 1]
time_pred = (time_proba >= 0.5).astype(int)

time_auc = roc_auc_score(
    y_test_time,
    time_proba
)

time_precision = precision_score(
    y_test_time,
    time_pred,
    zero_division=0
)

print("AFTER — Time-aware split")
print("-------------------------")
print("ROC-AUC:", round(time_auc, 4))
print("Precision:", round(time_precision, 4))

AFTER — Time-aware split
-------------------------
ROC-AUC: 0.9601
Precision: 0.7781


In [15]:
comparison = pd.DataFrame({
    "Evaluation": [
        "Week-5 reported result",
        "Random split",
        "Time-aware split"
    ],
    "ROC_AUC": [
        0.9606,
        random_auc,
        time_auc
    ],
    "Precision": [
        np.nan,
        random_precision,
        time_precision
    ]
})

display(comparison)

,Evaluation,ROC_AUC,Precision
0,Week-5 reported result,0.960600,NaN
1,Random split,0.949758,0.741033
2,Time-aware split,0.960124,0.778086


### Interpretation

The random split can produce an optimistic estimate because observations from the same client and content item can appear in both training and testing data.

The time-aware split is more representative of the intended use case because the model learns from earlier observations and is evaluated on later observations.

Therefore, the time-aware result should be treated as the stronger validation result for this forecasting-style task.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*


==>The goal of this audit is to check whether any feature contains information that would only be available after the prediction point.

The prediction point is the current report date.

The target uses next-day Google Search Console impressions.

Therefore, next-day variables must never be included as model features.

In [16]:
# Check for obvious future-information columns

future_keywords = [
    "next",
    "target",
    "future"
]

possible_leakage = [
    col for col in feature_cols
    if any(word in col.lower() for word in future_keywords)
]

print("Potential future-information features:")
print(possible_leakage)

if len(possible_leakage) == 0:
    print("\nNo obvious next/target/future columns are used as features.")
else:
    print("\nWARNING: Review these features before finalizing the model.")

Potential future-information features:
[]

No obvious next/target/future columns are used as features.


In [17]:
# Confirm target-related columns are not in the feature set

forbidden_features = [
    "next_gsc_impressions",
    "next_date",
    "target",
    "next_day_observed"
]

leakage_found = [
    col for col in forbidden_features
    if col in feature_cols
]

print("Forbidden columns found in feature set:", leakage_found)

assert len(leakage_found) == 0, \
    "Potential target leakage detected!"

Forbidden columns found in feature set: []


In [18]:
# Check whether duplicate client-content-date observations exist

duplicate_count = model_df.duplicated(
    subset=[
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ]
).sum()

print(
    "Duplicate client-content-date rows:",
    duplicate_count
)

Duplicate client-content-date rows: 0


In [19]:
# Show the date ranges used for training and testing

print("TRAINING PERIOD")
print(
    train_time["report_date"].min(),
    "to",
    train_time["report_date"].max()
)

print("\nTEST PERIOD")
print(
    test_time["report_date"].min(),
    "to",
    test_time["report_date"].max()
)

TRAINING PERIOD
2025-01-27 00:00:00 to 2025-03-24 00:00:00

TEST PERIOD
2025-03-25 00:00:00 to 2025-04-17 00:00:00


### Leakage conclusion

No explicit next-day or target-derived variables were included in the final feature set.

The evaluation also uses a chronological split, so observations from the future test period are not used to train the model.

This makes the time-aware evaluation more appropriate than a random split for this prediction task.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

===>## 4. Claim rewrite

The original model result should not be described as proof that the model will always predict future content performance correctly.

A safer claim focuses on what was actually measured on the available warehouse sample and validation design.

In [20]:
print("Original-style claim:")
print(
    "The model accurately predicts which pages will perform well."
)

print("\nSafer claim:")
print(
    f"On the evaluated FlyRank warehouse sample, the Random Forest "
    f"achieved a time-aware ROC-AUC of {time_auc:.4f}. "
    f"This indicates useful ranking signal under the tested validation setup, "
    f"but does not establish performance on unseen clients or future warehouse data."
)

Original-style claim:
The model accurately predicts which pages will perform well.

Safer claim:
On the evaluated FlyRank warehouse sample, the Random Forest achieved a time-aware ROC-AUC of 0.9601. This indicates useful ranking signal under the tested validation setup, but does not establish performance on unseen clients or future warehouse data.


In [21]:
# Show the most important features from the honest time-aware model

importance_df = pd.DataFrame({
    "feature": X_train_time.columns,
    "importance": time_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance_df.head(15))

,feature,importance
4,gsc_impressions,0.624533
6,gsc_sum_position,0.279034
7,gsc_avg_position,0.068426
5,gsc_clicks,0.028007
2,gsc_data_available,0.000000
0,client_has_gsc,0.000000
1,client_has_ga4,0.000000
3,ga4_data_available,0.000000
8,ga4_pageviews,0.000000
9,ga4_sessions,0.000000


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.